In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import imageio
import mediapipe as mp
import cv2
import os
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
from tensorflow.keras.utils import to_categorical  # For converting labels to categorical format (one-hot encoding)
from tensorflow.keras.models import Sequential  # For defining the neural network architecture
from tensorflow.keras.layers import LSTM, Dense  # For adding LSTM and Dense layers to the model
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping  # For logging training progress for TensorBoard
from tensorflow.keras.models import load_model  # For loading pre-trained models


In [2]:
# gloabl variables for label mapping
# removed j and z from this initial static classifier
static_label_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
static_label_map = {label: num for num, label in enumerate(static_label_names)}

In [3]:
# load train, validation, and test data

def load_split(split):
    images = []
    labels = []
    for label in static_label_names:
        image_paths = sorted(glob.glob(f'datasets/SignImages/{split}/{label}/*.jpg'))
        for path in image_paths:
            image = imageio.imread(path)
            images.append(image)
            labels.append(label)
    return np.array(images), np.array(labels)

train_images, train_labels = load_split('train')
val_images, val_labels = load_split('validation')
test_images, test_labels = load_split('test')

C:\Users\Ida\AppData\Local\Temp\ipykernel_38220\1722234016.py:9: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(path)


In [4]:
# extract landmarks from images using MediaPipe Hands (see examples/hands.py)

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

def extract_landmarks(images, labels):
    landmarks = []

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=2,
        min_detection_confidence=0.5) as hands:

        for image in images:
            results = hands.process(image)
            left = np.zeros(63)
            right = np.zeros(63)

            if results.multi_hand_landmarks and results.multi_handedness:
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]).flatten()
                    if handedness.classification[0].label == 'Left':
                        left = coords
                    else:
                        right = coords
                features = np.concatenate([left, right])
                landmarks.append(features)
            else:
                landmarks.append(None)  # no hand detected

    # remove entries with no landmarks detected
    df = pd.DataFrame({
        'landmarks': landmarks,
        'label': labels
    })
    df = df[df['landmarks'].notna()]

    detected_landmarks = np.array(df['landmarks'].tolist())
    detected_labels = np.array(df['label'].values)

    ### TODO: when moving to dynamic signs, need to remove this repeat   ###
    ### repeat landmarks to create sequences of length 30 for LSTM input ###
    detected_landmarks = np.repeat(detected_landmarks[:, np.newaxis, :], 30, axis=1)

    return detected_landmarks, detected_labels

train_landmarks, train_labels = extract_landmarks(train_images, train_labels)
val_landmarks, val_labels = extract_landmarks(val_images, val_labels)

train_labels_int = np.array([static_label_map[l] for l in train_labels])
train_labels_cat = to_categorical(train_labels_int, num_classes=len(static_label_names))

val_labels_int = np.array([static_label_map[l] for l in val_labels])
val_labels_cat = to_categorical(val_labels_int, num_classes=len(static_label_names))


In [5]:
# define LSTM model for static sign classification
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30, 126)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(len(static_label_names), activation='softmax'))

model.compile(optimizer='Adam',
              # Use 'categorical_crossentropy' as the loss function, suitable for multi-class classification problems
              loss='categorical_crossentropy',
              # Track 'categorical_accuracy' during training to evaluate how well the model is classifying the correct action
              metrics=['categorical_accuracy'])

log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)
early_stopping = EarlyStopping(monitor='val_categorical_accuracy', patience=3, restore_best_weights=True)

# start training the model
model.fit(train_landmarks, train_labels_cat, validation_data=(val_landmarks, val_labels_cat), epochs=200, callbacks=[tb_callback, early_stopping])

C:\Users\Ida\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
590/590 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - categorical_accuracy: 0.6378 - loss: 1.0839 - val_categorical_accuracy: 0.5463 - val_loss: 1.8240
Epoch 2/200
590/590 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.7635 - loss: 0.7568 - val_categorical_accuracy: 0.9294 - val_loss: 0.2235
Epoch 3/200
590/590 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.9021 - loss: 0.3161 - val_categorical_accuracy: 0.8389 - val_loss: 0.4407
Epoch 4/200
590/590 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.9089 - loss: 0.2580 - val_categorical_accuracy: 0.9349 - val_loss: 0.1722
Epoch 5/200
590/590 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.9485 - loss: 0.1436 - val_categorical_accuracy: 0.9949 - val_loss: 0.0207
Epoch 6/200
590/590 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.8976 - loss: 34.2033 - val_categorical_accuracy: 0.9681 - val_loss: 0.1348
Epoch 7/200
590/590 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_

In [6]:
# evaluate the model on the test set
test_images, test_labels = load_split('test')

test_landmarks, test_labels = extract_landmarks(test_images, test_labels)
test_labels_int = np.array([static_label_map[l] for l in test_labels])
test_labels_cat = to_categorical(test_labels_int, num_classes=len(static_label_names))

test_loss, test_acc = model.evaluate(test_landmarks, test_labels_cat)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

C:\Users\Ida\AppData\Local\Temp\ipykernel_38220\1722234016.py:9: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(path)


74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - categorical_accuracy: 1.0000 - loss: 0.0010
Test Loss: 0.0010
Test Accuracy: 1.0000


In [7]:
# save the trained model
model.save('asl_model.keras')

In [8]:
# load the model and run real-time inference on webcam feed
# model = load_model('asl_model.keras')
cap = cv2.VideoCapture(0)

sequence = []
threshold = 0.8

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture image.")
        break

    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=2,
        min_detection_confidence=0.5) as hands:
        
        results = hands.process(frame_rgb)

        left = np.zeros(63)
        right = np.zeros(63)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]).flatten()
                if handedness.classification[0].label == 'Left':
                    left = coords
                else:
                    right = coords

            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        features = np.concatenate([left, right])
        sequence.append(features)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            input_data = np.expand_dims(sequence, axis=0)
            prediction = model.predict(input_data, verbose=0)
            confidence = np.max(prediction)
            predicted_index = np.argmax(prediction)
            predicted_label = static_label_names[predicted_index]

            # display on frame
            if confidence > threshold:
                cv2.putText(frame, f'{predicted_label} ({confidence:.2f})',
                        (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.5, (0, 255, 0), 3)
            else:
                cv2.putText(frame, 'Unknown',
                        (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.5, (0, 0, 255), 3)

    cv2.imshow('ASL Interpreter', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()